# PageIndex-Guided Hybrid Retrieval — Experiment

**Not one of the thesis's three canonical pipelines** — this is a fourth,
exploratory architecture that combines pieces of the other two:

1. Take the page(s) PageIndex's navigation agent (Stage 3 of the vectorless
   RAG pipeline) already picked for a question — no new LLM navigation calls.
2. Restrict the candidate chunk pool to only chunks on those pages.
3. Run vector RAG's dense+BM25 hybrid retrieval *within that restricted pool*
   (same math, same alpha, as the real vector RAG pipeline).
4. Score the result with Recall@k / MRR@k / NDCG@k against FinanceBench's
   gold evidence pages.

**Question this answers:** does letting PageIndex narrow down the search
space *before* hybrid retrieval runs help (less noise to search through) or
hurt (a wrong PageIndex pick now blocks hybrid retrieval from ever finding
the right chunk, even if hybrid alone would have found it)?

**Reuses, does not recompute:**
- PageIndex trees: `experiments/results/vectorless_rag_index_flash/` (Stage 2b
  of the vectorless notebook)
- PageIndex navigation output: `experiments/results/vectorless_rag_stage_navigation_expanded.jsonl`
  (Stage 3 of the vectorless notebook)
- Vector RAG's chunk index + dense embeddings: `experiments/results/vector_rag_index/`
  (Stage 4 of the vector RAG notebook)
- Vector RAG's expanded-query embeddings: `experiments/results/vector_rag_index/queries/`
  (Stage 5 of the vector RAG notebook)

**No new API calls at all** — navigation and embedding were already paid for
by the other two pipelines. This notebook is pure recombination + BM25 math
(free, CPU-only) + metrics. Safe/cheap to re-run as often as you want.

**A note on page numbering (read this before trusting any output below):**
PageIndex's `start_index`/`end_index` on a tree node are **1-indexed** PDF
page numbers, inclusive on both ends (see `fetch_node_text` in the
vectorless notebook: `page_texts[start_index - 1 : end_index]`). Vector RAG's
`page_num` column is **0-indexed** — `pymupdf4llm`'s raw `page_number` minus
1, chosen specifically to match FinanceBench's own `evidence_page_num`
convention (see the vector RAG notebook's Stage 4 walkthrough). Converting a
navigated node into a `page_num` filter therefore needs a `-1` on *both*
ends of the range — done explicitly in `navigated_pages()` below, not left
implicit, since silently mixing the two conventions would make every
downstream page comparison wrong by one and still *look* plausible.

---
## Stage 0 — Setup

In [40]:
import os, sys, json, time, traceback
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from google.colab import drive

drive.mount('/content/drive')

REPO_ROOT = Path('/content/drive/MyDrive/financebench_project')
if not (REPO_ROOT / "data" / "financebench_open_source.jsonl").exists():
    print(f"Repo not found at {REPO_ROOT} — cloning ...")
    !git clone https://github.com/shaliqsv/financebench-rag-thesis.git "{REPO_ROOT}"
else:
    print(f"Repo already present at {REPO_ROOT} — syncing to latest main ...")
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" checkout main
    !git -C "{REPO_ROOT}" pull origin main --no-rebase --no-edit

sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "experiments" / "results"

# Reused as-is from the other two pipelines — nothing new is written to any
# of these four paths by this notebook.
TREE_DIR_FLASH = RESULTS_DIR / "vectorless_rag_index_flash"
# Expanded-query navigation: the config the vectorless pipeline generated its answers from
NAVIGATION_PATH = RESULTS_DIR / "vectorless_rag_stage_navigation_expanded.jsonl"
INDEX_DIR = RESULTS_DIR / "vector_rag_index"
QUERIES_DIR = INDEX_DIR / "queries"
HYBRID_PATH = RESULTS_DIR / "vector_rag_stage_hybrid.jsonl"  # plain-hybrid baseline, for comparison at the end

# This experiment's own output — isolated from all three canonical pipelines'
# result files, per CLAUDE.md's "keep the three pipelines independent" rule.
# New filename (not ..._stage_retrieval.jsonl): the loop skips ids already in OUT_PATH, so reusing the
# old file would mix results from the original navigation with the expanded one.
OUT_PATH = RESULTS_DIR / "pageindex_hybrid_expanded_stage_retrieval.jsonl"

load_dotenv(REPO_ROOT / ".env", override=True)
GH_TOKEN = os.getenv("GH_TOKEN", "")
JINA_API_KEY = os.getenv("JINA_API_KEY", "")
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
print(f"REPO_ROOT: {REPO_ROOT}")
print("JINA_API_KEY:", "ok" if JINA_API_KEY else "MISSING — the rerank stage below will fail")
print("GROQ_API_KEY:", "ok" if GROQ_API_KEY else "MISSING — the scoring stage (LLM judge) will fail")
print("GH_TOKEN:", "ok" if GH_TOKEN else "MISSING — sync-to-GitHub cell at the end will fail to push")

for label, path in [("TREE_DIR_FLASH", TREE_DIR_FLASH), ("NAVIGATION_PATH", NAVIGATION_PATH),
                     ("INDEX_DIR", INDEX_DIR), ("QUERIES_DIR", QUERIES_DIR)]:
    exists = path.exists()
    print(f"{label:<14}: {path}  {'ok' if exists else 'MISSING'}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Repo already present at /content/drive/MyDrive/financebench_project — syncing to latest main ...
Already on 'main'
Your branch is up to date with 'origin/main'.
From https://github.com/shaliqsv/financebench-rag-thesis
 * branch            main       -> FETCH_HEAD
Already up to date.
REPO_ROOT: /content/drive/MyDrive/financebench_project
JINA_API_KEY: ok
GROQ_API_KEY: ok
GH_TOKEN: ok
TREE_DIR_FLASH: /content/drive/MyDrive/financebench_project/experiments/results/vectorless_rag_index_flash  ok
NAVIGATION_PATH: /content/drive/MyDrive/financebench_project/experiments/results/vectorless_rag_stage_navigation_expanded.jsonl  ok
INDEX_DIR     : /content/drive/MyDrive/financebench_project/experiments/results/vector_rag_index  ok
QUERIES_DIR   : /content/drive/MyDrive/financebench_project/experiments/results/vector_rag_index/queries  ok


In [12]:
!git -C "{REPO_ROOT}" config user.email "shaliqv25@gmail.com"
!git -C "{REPO_ROOT}" config user.name "shaliqsv"

if GH_TOKEN:
    !git -C "{REPO_ROOT}" remote set-url origin https://{GH_TOKEN}@github.com/shaliqsv/financebench-rag-thesis.git
    print("git identity set, remote configured with token auth")
else:
    print("WARNING: GH_TOKEN missing from .env — sync-to-GitHub cell at the end will fail to push.")

git identity set, remote configured with token auth


In [2]:
# Retrieval stages only read parquet/npy/json files other pipelines already
# produced, plus BM25 (CPU, free). The rerank stage (Jina, plain REST call) and
# the generation/scoring stages at the end (litellm -> Gemini, groq -> judge)
# do call APIs -- litellm and groq are installed for those.
%pip install -q pandas numpy rank_bm25 pyarrow requests litellm groq

---
## Stage 1 — Load FinanceBench data

In [3]:
df_questions = pd.read_json(DATA_DIR / "financebench_open_source.jsonl", lines=True)
df_meta = pd.read_json(DATA_DIR / "financebench_document_information.jsonl", lines=True)
df = pd.merge(df_questions, df_meta, on=["doc_name", "company"])

print(f"Total questions : {len(df)}")
print(f"Unique documents: {df.doc_name.nunique()}")
df[["financebench_id", "doc_name", "question_type", "question"]].head(3)

Total questions : 150
Unique documents: 84


,financebench_id,doc_name,question_type,question
0,financebench_id_03029,3M_2018_10K,metrics-generated,What is the FY2018 capital expenditure amount ...
1,financebench_id_04672,3M_2018_10K,metrics-generated,Assume that you are a public equities analyst....
2,financebench_id_00499,3M_2022_10K,domain-relevant,Is 3M a capital-intensive business based on FY...


---
## Stage 2 — Shared read helpers + PageIndex tree lookup

`_load_jsonl` / `_load_stage_records` / `_load_completed_ids` are copied
verbatim from the vector RAG and vectorless RAG notebooks (same tiny
functions, same names) — not imported, following this project's existing
convention of keeping each pipeline notebook self-contained top-to-bottom
(see the vector RAG notebook's own note on this). `evaluation/retrieval_metrics.py`
*is* imported below, since that module is the one piece already meant to be
shared across pipelines.

In [4]:
def _load_jsonl(path) -> list[dict]:
    if not path.exists():
        return []
    with path.open() as f:
        return [json.loads(line) for line in f if line.strip()]


def _load_completed_ids(path) -> set:
    return {r["financebench_id"] for r in _load_jsonl(path)}


def _load_stage_records(path) -> dict:
    return {r["financebench_id"]: r for r in _load_jsonl(path)}


def tree_path(tree_dir, doc_name):
    return tree_dir / f"{doc_name}_tree.json"


def is_tree_built(tree_dir, doc_name) -> bool:
    return tree_path(tree_dir, doc_name).exists()


def load_tree(tree_dir, doc_name) -> dict:
    return json.loads(tree_path(tree_dir, doc_name).read_text())


def create_node_mapping(structure) -> dict:
    """node_id -> node, flattened from PageIndex's nested tree. Copied from
    pageindex.utils.create_node_mapping (external/PageIndex/pageindex/utils.py)
    rather than pip-installing the whole pageindex package (and its litellm/
    PyPDF2/pyyaml dependencies) just for this one 8-line function."""
    mapping = {}

    def _traverse(nodes):
        for node in nodes:
            if node.get("node_id"):
                mapping[node["node_id"]] = node
            if node.get("nodes"):
                _traverse(node["nodes"])

    _traverse(structure)
    return mapping


def navigated_pages(node_ids, node_map) -> set:
    """Union of 0-indexed page_nums (vector RAG's convention) covered by the
    given PageIndex node_ids. See the page-numbering note in the title cell
    above for why the -1 is here."""
    pages = set()
    for node_id in node_ids:
        node = node_map.get(node_id)
        if node is None:
            continue  # navigation occasionally returns a stale/hallucinated node_id
        for one_indexed_page in range(node["start_index"], node["end_index"] + 1):
            pages.add(one_indexed_page - 1)
    return pages

---
## Stage 3 — Hybrid retrieval building blocks (copied from vector RAG)

`bm25_tokenize`, `build_bm25`, `load_index`, `hybrid_retrieve` are the exact
same functions as the vector RAG notebook's Stage 6, same `ALPHA_DENSE=0.85`
(Kim et al.) — so any difference in results comes from *which chunks get
handed to hybrid retrieval*, not from a different retrieval algorithm.
`query_paths` / `is_query_embedded` read the query vectors Stage 5 of vector
RAG already saved — no re-embedding here.

In [5]:
import re
from rank_bm25 import BM25Okapi

ALPHA_DENSE = 0.85  # dense weight, per Kim et al. — matches vector RAG exactly

_BM25_TOKEN_RE = re.compile(r"[a-z0-9$][a-z0-9.,%$-]*")


def bm25_tokenize(text: str) -> list[str]:
    return _BM25_TOKEN_RE.findall(text.lower())


def build_bm25(chunks_df) -> BM25Okapi:
    return BM25Okapi([bm25_tokenize(t) for t in chunks_df["text"]])


def index_paths(index_dir, doc_name):
    return index_dir / f"{doc_name}_chunks.parquet", index_dir / f"{doc_name}_dense.npy"


def load_index(index_dir, doc_name):
    chunks_path, dense_path = index_paths(index_dir, doc_name)
    chunks_df = pd.read_parquet(chunks_path)
    dense_embeddings = np.load(dense_path)
    assert len(chunks_df) == dense_embeddings.shape[0], f"{doc_name}: chunks/embeddings length mismatch"
    return chunks_df, dense_embeddings


def hybrid_retrieve(chunks_df, dense_embeddings, bm25_index, query_vec, expanded_query, top_k=20, alpha=ALPHA_DENSE):
    dense_scores = dense_embeddings @ query_vec
    bm25_scores_raw = bm25_index.get_scores(bm25_tokenize(expanded_query))

    bm25_min, bm25_max = bm25_scores_raw.min(), bm25_scores_raw.max()
    bm25_scores_norm = (bm25_scores_raw - bm25_min) / (bm25_max - bm25_min + 1e-9)

    hybrid_scores = alpha * dense_scores + (1 - alpha) * bm25_scores_norm

    top_idx = np.argsort(-hybrid_scores)[:top_k]
    top = chunks_df.iloc[top_idx].copy()
    top["hybrid_score"] = hybrid_scores[top_idx]
    top["dense_score"] = dense_scores[top_idx]
    top["bm25_score"] = bm25_scores_raw[top_idx]
    return top


def query_paths(queries_dir, financebench_id):
    return (queries_dir / f"{financebench_id}__expanded.npy", queries_dir / f"{financebench_id}__expanded.json")


def is_query_embedded(queries_dir, financebench_id) -> bool:
    vec_path, meta_path = query_paths(queries_dir, financebench_id)
    return vec_path.exists() and meta_path.exists()

---
## Stage 4 — The new piece: restrict to navigated pages, then hybrid-retrieve

`restrict_to_pages` filters `chunks_df` down to rows on the navigated pages
— and must filter `dense_embeddings` with the *same* boolean mask, converted
to a numpy array, so the two stay row-aligned. Filtering the DataFrame and
the embeddings array separately (e.g. re-deriving the mask twice) is exactly
the kind of thing that can silently desync them — mask once, reuse it for
both.

In [6]:
def restrict_to_pages(chunks_df, dense_embeddings, page_nums):
    mask = chunks_df["page_num"].isin(page_nums)
    return chunks_df[mask].reset_index(drop=True), dense_embeddings[mask.to_numpy()]

In [10]:
from evaluation.retrieval_metrics import compute_retrieval_metrics, aggregate_retrieval_metrics, RetrievalMetrics, K_VALUES


def run_pageindex_hybrid_all(df, tree_dir, index_dir, queries_dir, navigation_path, out_path, top_k=20, alpha=ALPHA_DENSE):
    """Resumable, same pattern as vector RAG's run_hybrid_retrieval_all: one
    line per question appended to out_path, already-done financebench_ids
    skipped on re-run."""
    out_path.parent.mkdir(parents=True, exist_ok=True)
    errors_log = out_path.parent / "pageindex_hybrid_errors.log"
    navigation_records = _load_stage_records(navigation_path)
    completed = _load_completed_ids(out_path)

    tree_cache, node_map_cache, doc_cache = {}, {}, {}
    results = {"done": [], "skipped": [], "failed": [], "missing_navigation": [], "missing_tree": [], "missing_query": []}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        doc_name = row.doc_name

        if fb_id in completed:
            results["skipped"].append(fb_id)
            continue
        if fb_id not in navigation_records:
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED — no navigation output yet")
            results["missing_navigation"].append(fb_id)
            continue
        if not is_tree_built(tree_dir, doc_name):
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED — {doc_name} has no PageIndex tree yet")
            results["missing_tree"].append(fb_id)
            continue
        if not is_query_embedded(queries_dir, fb_id):
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED — query not embedded yet")
            results["missing_query"].append(fb_id)
            continue

        print(f"[{i}/{len(df)}] {fb_id}: retrieving ...")
        try:
            if doc_name not in tree_cache:
                tree_cache[doc_name] = load_tree(tree_dir, doc_name)
                node_map_cache[doc_name] = create_node_mapping(tree_cache[doc_name]["structure"])
            if doc_name not in doc_cache:
                doc_cache[doc_name] = load_index(index_dir, doc_name)
            chunks_df, dense_embeddings = doc_cache[doc_name]

            node_ids = navigation_records[fb_id]["node_ids"]
            page_nums = navigated_pages(node_ids, node_map_cache[doc_name])

            restricted_df, restricted_embeddings = restrict_to_pages(chunks_df, dense_embeddings, page_nums)
            fell_back = len(restricted_df) == 0
            if fell_back:
                # Navigated pages produced zero chunks (e.g. a near-blank
                # page, or a page_num that doesn't exist in this doc's
                # chunk index) -- fall back to the whole document rather
                # than handing BM25/hybrid_retrieve an empty candidate set.
                restricted_df, restricted_embeddings = chunks_df, dense_embeddings

            bm25_index = build_bm25(restricted_df)

            vec_path, meta_path = query_paths(queries_dir, fb_id)
            query_vec = np.load(vec_path)
            expanded_query = json.loads(meta_path.read_text())["query_text"]

            top_hybrid = hybrid_retrieve(restricted_df, restricted_embeddings, bm25_index, query_vec, expanded_query,
                                          top_k=top_k, alpha=alpha)

            gold_pages = [e["evidence_page_num"] for e in row.evidence]
            metrics = compute_retrieval_metrics(
                ranked_pages=top_hybrid["page_num"].tolist(), gold_pages=gold_pages,
                financebench_id=fb_id, doc_name=doc_name, stage="pageindex_hybrid_top20",
            )

            record = {
                "financebench_id": fb_id, "doc_name": doc_name,
                "navigated_node_ids": node_ids, "navigated_pages": sorted(page_nums),
                "n_candidates": len(restricted_df), "fell_back_to_full_doc": fell_back,
                "chunk_ids": top_hybrid["chunk_id"].tolist(), "page_nums": top_hybrid["page_num"].tolist(),
                "recall_at_k": metrics.recall_at_k, "mrr_at_k": metrics.mrr_at_k, 
                "timestamp": time.time(),
            }
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")

            fallback_note = " (fell back to full doc)" if fell_back else ""
            print(f"    {len(restricted_df)} candidates from {len(page_nums)} navigated pages{fallback_note} "
                  f"— recall@5={metrics.recall_at_k[5]:.2f}")
            results["done"].append(fb_id)
        except Exception as e:
            print(f"    FAILED: {e}")
            with errors_log.open("a") as f:
                f.write(f"{fb_id}\t{e}\n{traceback.format_exc()}\n---\n")
            results["failed"].append(fb_id)

    print(f"\nSummary: {len(results['done'])} done, {len(results['skipped'])} already done, "
          f"{len(results['failed'])} failed, {len(results['missing_navigation'])} missing navigation, "
          f"{len(results['missing_tree'])} missing tree, {len(results['missing_query'])} missing query embedding")
    return results

---
### Sanity check on a handful of questions first

Same reasoning as the project brief's own timeline (get it working on ~10-15
questions before the full 150): run on the first 5 rows of `df`, read the
printed candidate counts and recall/ndcg by eye, *then* run the full batch
below.

In [8]:
_smoke_test_results = run_pageindex_hybrid_all(
    df=df.head(5), tree_dir=TREE_DIR_FLASH, index_dir=INDEX_DIR, queries_dir=QUERIES_DIR,
    navigation_path=NAVIGATION_PATH, out_path=OUT_PATH,
)

[1/5] financebench_id_03029: retrieving ...
    322 candidates from 114 navigated pages — recall@5=0.00
[2/5] financebench_id_04672: retrieving ...
    217 candidates from 77 navigated pages — recall@5=1.00
[3/5] financebench_id_00499: retrieving ...
    301 candidates from 112 navigated pages — recall@5=0.00
[4/5] financebench_id_01226: retrieving ...
    285 candidates from 106 navigated pages — recall@5=1.00
[5/5] financebench_id_01865: retrieving ...
    285 candidates from 106 navigated pages — recall@5=0.00

Summary: 5 done, 0 already done, 0 failed, 0 missing navigation, 0 missing tree, 0 missing query embedding


### Full run — all 150 questions

In [9]:
pageindex_hybrid_results = run_pageindex_hybrid_all(
    df=df, tree_dir=TREE_DIR_FLASH, index_dir=INDEX_DIR, queries_dir=QUERIES_DIR,
    navigation_path=NAVIGATION_PATH, out_path=OUT_PATH,
)

[6/150] financebench_id_00807: retrieving ...
    162 candidates from 55 navigated pages — recall@5=1.00
[7/150] financebench_id_00941: retrieving ...
    14 candidates from 8 navigated pages — recall@5=1.00
[8/150] financebench_id_01858: retrieving ...
    83 candidates from 26 navigated pages — recall@5=1.00
[9/150] financebench_id_02987: retrieving ...
    136 candidates from 60 navigated pages — recall@5=1.00
[10/150] financebench_id_07966: retrieving ...
    197 candidates from 89 navigated pages — recall@5=0.50
[11/150] financebench_id_04735: retrieving ...
    98 candidates from 25 navigated pages — recall@5=1.00
[12/150] financebench_id_07507: retrieving ...
    41 candidates from 18 navigated pages — recall@5=1.00
[13/150] financebench_id_03856: retrieving ...
    9 candidates from 4 navigated pages — recall@5=1.00
[14/150] financebench_id_00438: retrieving ...
    59 candidates from 28 navigated pages — recall@5=1.00
[15/150] financebench_id_00591: retrieving ...
    118 cand

---
## Stage 5 — Summarize

`aggregate_retrieval_metrics` is the shared function from
`evaluation/retrieval_metrics.py`. One gotcha: JSON always serializes dict
keys as strings, so `recall_at_k`/`mrr_at_k`/`ndcg_at_k` come back from
`OUT_PATH` as `{"5": ..., "10": ..., "15": ...}` — but `aggregate_retrieval_metrics`
indexes them with the *int* values from `K_VALUES` (`r.recall_at_k[5]`, not
`r.recall_at_k["5"]`). The `int(k)` conversion below exists for exactly that
reason — skip it and every lookup inside `aggregate_retrieval_metrics` raises
a `KeyError`.

In [11]:
def _to_retrieval_metrics(record) -> RetrievalMetrics:
    return RetrievalMetrics(
        financebench_id=record["financebench_id"], doc_name=record["doc_name"], stage="pageindex_hybrid",
        recall_at_k={int(k): v for k, v in record["recall_at_k"].items()},
        mrr_at_k={int(k): v for k, v in record["mrr_at_k"].items()},
        
    )


pageindex_hybrid_records = _load_jsonl(OUT_PATH)
pageindex_hybrid_metrics = [_to_retrieval_metrics(r) for r in pageindex_hybrid_records]
pageindex_hybrid_summary = aggregate_retrieval_metrics(pageindex_hybrid_metrics)
pageindex_hybrid_summary

{'n_questions': 150,
 'recall_at_k': {5: 0.7466666666666667, 10: 0.7988888888888889, 15: 0.85},
 'mrr_at_k': {5: 0.5962222222222222,
  10: 0.6030343915343915,
  15: 0.6058338143338143}}

### How often did it fall back to the full document?

If this number is high, PageIndex's navigated pages are frequently
producing zero matching chunks — worth checking `navigated_pages()`'s page
math (or PageIndex's own node ranges) before trusting the metrics above,
since a high fallback rate means this "PageIndex-guided" run is quietly
behaving like plain hybrid retrieval for a large chunk of the 150
questions.

In [12]:
n_fell_back = sum(1 for r in pageindex_hybrid_records if r["fell_back_to_full_doc"])
print(f"{n_fell_back}/{len(pageindex_hybrid_records)} questions fell back to the full document")

0/150 questions fell back to the full document


### Compare against plain hybrid retrieval (no PageIndex narrowing)

`HYBRID_PATH` is vector RAG's own Stage 6 output — same hybrid retrieval
math, run over the *whole* document instead of a PageIndex-narrowed subset.
Only Recall@k / MRR@k are compared here (that file predates
`evaluation/retrieval_metrics.py`'s NDCG addition, so it was never computed
there) — re-run vector RAG's Stage 6 through the shared module first if
NDCG needs to be in this comparison too.

In [13]:
plain_hybrid_records = _load_jsonl(HYBRID_PATH)
plain_hybrid_by_id = {r["financebench_id"]: r for r in plain_hybrid_records}

# Only compare on questions both runs actually have output for, so a
# still-in-progress HYBRID_PATH (or a still-in-progress OUT_PATH) doesn't
# quietly skew the average toward whichever one has more coverage.
shared_ids = [r["financebench_id"] for r in pageindex_hybrid_records if r["financebench_id"] in plain_hybrid_by_id]
print(f"Comparing on {len(shared_ids)} questions with output in both runs\n")

for k in K_VALUES:
    pageindex_recall = sum(r["recall_at_k"][str(k)] for r in pageindex_hybrid_records if r["financebench_id"] in shared_ids) / len(shared_ids)
    plain_recall = sum(plain_hybrid_by_id[fb_id]["recall_at_k"][str(k)] for fb_id in shared_ids) / len(shared_ids)
    print(f"recall@{k:<3} pageindex_hybrid={pageindex_recall:.3f}   plain_hybrid={plain_recall:.3f}")

Comparing on 150 questions with output in both runs

recall@5   pageindex_hybrid=0.747   plain_hybrid=0.747
recall@10  pageindex_hybrid=0.799   plain_hybrid=0.832
recall@15  pageindex_hybrid=0.850   plain_hybrid=0.857


---
## Navigation hit rate vs. hybrid recall

Hybrid can only find a gold page if navigation put it in the candidate pool
(unless the run fell back to the whole document). So the navigation hit rate
is the ceiling for Recall@k here. `nav_hit` uses the same rule as the vectorless
notebook's `recall_df`: any gold page covered by a navigated page.

In [15]:
records = _load_jsonl(OUT_PATH)
rows = []
for r in records:
    gold = {e["evidence_page_num"] for e in df.loc[df.financebench_id == r["financebench_id"]].iloc[0].evidence}
    rows.append({
        "financebench_id": r["financebench_id"], "doc_name": r["doc_name"],
        "nav_hit": bool(gold & set(r["navigated_pages"])),
        "fell_back": r["fell_back_to_full_doc"], "n_candidates": r["n_candidates"],
        **{f"recall@{k}": r["recall_at_k"][str(k)] for k in K_VALUES},
    })
analysis = pd.DataFrame(rows)

print(f"Navigation hit rate (ceiling for hybrid): {analysis.nav_hit.mean():.3f}  ({len(analysis)} questions)")
print(f"Median candidate chunks after narrowing : {analysis.n_candidates.median():.0f}\n")
print("Mean recall@k by whether navigation hit:")
print(analysis.groupby("nav_hit")[[f"recall@{k}" for k in K_VALUES]].mean().round(3))

Navigation hit rate (ceiling for hybrid): 0.940  (150 questions)
Median candidate chunks after narrowing : 93

Mean recall@k by whether navigation hit:
         recall@5  recall@10  recall@15
nav_hit                                
False       0.000       0.00      0.000
True        0.794       0.85      0.904


---
## Stage 6 — Rerank the narrowed hybrid top-20 down to top-10

Same reranker and same funnel as vector RAG's Stage 7: Jina
`jina-reranker-v2-base-multilingual`, hybrid top 20 in, top 10 out, query =
the expanded query. `_jina_rerank` / `rerank` are copied from the vector RAG
notebook (self-contained-notebook convention). The hybrid top-20 comes from
this notebook's own `OUT_PATH`, so the candidate pool is the PageIndex-narrowed
one, not the whole document.

**Costs API calls** (Jina), unlike everything above — 150 calls, one per
question. Token usage is saved in each record (`rerank_tokens`); Jina's price
isn't confirmed in this project yet, so no dollar figure is computed here.
Resumable: questions already in the output file are skipped.

Recall@15 is identical to Recall@10 here, since only 10 chunks come out.

In [ ]:
import requests
from functools import wraps

JINA_RERANK_URL = "https://api.jina.ai/v1/rerank"
JINA_RERANK_MODEL = "jina-reranker-v2-base-multilingual"
JINA_TOKEN_BUDGET = 9000   # same defensive cap as vector RAG (not verified against Jina's real per-request limit)
RERANK_OUT_PATH = RESULTS_DIR / "pageindex_hybrid_expanded_stage_rerank.jsonl"


def with_retry(max_retries=5, base_delay=10.0, max_delay=120.0):
    """Same exponential-backoff-on-429 helper as vector RAG's, trimmed to what this stage needs."""
    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            delay, last_exc = base_delay, None
            for attempt in range(max_retries):
                try:
                    return fn(*args, **kwargs)
                except Exception as e:
                    last_exc = e
                    if any(m in str(e).lower() for m in ("429", "rate limit", "quota")):
                        print(f"    rate limited, backing off {delay:.0f}s ...")
                        time.sleep(delay)
                        delay = min(delay * 2, max_delay)
                    elif attempt == 0:
                        time.sleep(2.0)   # one short retry for a network blip
                    else:
                        raise
            raise RuntimeError(f"{fn.__name__} failed after {max_retries} retries") from last_exc
        return wrapper
    return decorator


@with_retry()
def _jina_rerank(jina_api_key, query, documents, model, top_k):
    resp = requests.post(
        JINA_RERANK_URL,
        headers={"Authorization": f"Bearer {jina_api_key}", "Content-Type": "application/json"},
        json={"model": model, "query": query, "documents": documents, "top_n": top_k},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()


def rerank(jina_api_key, query, candidates_df, top_k=10, model=JINA_RERANK_MODEL):
    """Returns (reranked_df, total_tokens). Trims the lowest-ranked candidates first if the
    set would exceed JINA_TOKEN_BUDGET (candidates arrive in hybrid-score order)."""
    trimmed = candidates_df
    if "token_count" in trimmed.columns:
        within_budget = trimmed["token_count"].cumsum() <= JINA_TOKEN_BUDGET
        if not within_budget.all():
            trimmed = trimmed.iloc[:max(1, within_budget.sum())]
    result = _jina_rerank(jina_api_key, query, trimmed["text"].tolist(), model, top_k)
    order = [r["index"] for r in result["results"]]
    top = trimmed.iloc[order].copy()
    top["rerank_score"] = [r["relevance_score"] for r in result["results"]]
    return top, result.get("usage", {}).get("total_tokens", 0)


def run_pageindex_hybrid_rerank_all(df, index_dir, queries_dir, hybrid_in_path, out_path, top_k=10):
    if not JINA_API_KEY:
        raise RuntimeError("JINA_API_KEY missing — fill it in .env")
    out_path.parent.mkdir(parents=True, exist_ok=True)
    errors_log = out_path.parent / "pageindex_hybrid_rerank_errors.log"
    hybrid_records = _load_stage_records(hybrid_in_path)
    completed = _load_completed_ids(out_path)
    doc_cache = {}
    results = {"done": [], "skipped": [], "failed": [], "missing_hybrid": []}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            results["skipped"].append(fb_id)
            continue
        if fb_id not in hybrid_records:
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED — no narrowed hybrid result yet")
            results["missing_hybrid"].append(fb_id)
            continue

        print(f"[{i}/{len(df)}] {fb_id}: reranking ...")
        try:
            hyb = hybrid_records[fb_id]
            if row.doc_name not in doc_cache:
                chunks_df, _ = load_index(index_dir, row.doc_name)
                doc_cache[row.doc_name] = chunks_df.set_index("chunk_id")
            candidates_df = doc_cache[row.doc_name].loc[hyb["chunk_ids"]].reset_index()

            _, meta_path = query_paths(queries_dir, fb_id)
            expanded_query = json.loads(meta_path.read_text())["query_text"]

            top_rerank, rerank_tokens = rerank(JINA_API_KEY, expanded_query, candidates_df, top_k=top_k)

            gold_pages = [e["evidence_page_num"] for e in row.evidence]
            metrics = compute_retrieval_metrics(
                ranked_pages=top_rerank["page_num"].tolist(), gold_pages=gold_pages,
                financebench_id=fb_id, doc_name=row.doc_name, stage=f"pageindex_hybrid_rerank_top{top_k}",
            )
            record = {
                "financebench_id": fb_id, "doc_name": row.doc_name,
                "chunk_ids": top_rerank["chunk_id"].tolist(), "page_nums": top_rerank["page_num"].tolist(),
                "rerank_scores": top_rerank["rerank_score"].tolist(), "rerank_tokens": rerank_tokens,
                "recall_at_k": metrics.recall_at_k, "mrr_at_k": metrics.mrr_at_k, "timestamp": time.time(),
            }
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
            print(f"    done — recall@10={metrics.recall_at_k[10]:.2f}")
            results["done"].append(fb_id)
        except Exception as e:
            print(f"    FAILED: {e}")
            with errors_log.open("a") as f:
                f.write(f"{fb_id}\t{e}\n{traceback.format_exc()}\n---\n")
            results["failed"].append(fb_id)

    print(f"\nRerank summary: {len(results['done'])} done, {len(results['skipped'])} already done, "
          f"{len(results['failed'])} failed, {len(results['missing_hybrid'])} missing hybrid result")
    return results

### Try 5 questions first, then the full run

Same habit as the hybrid stage above: read a few by eye before spending 150 API calls.

In [ ]:
_rerank_smoke = run_pageindex_hybrid_rerank_all(
    df=df.head(5), index_dir=INDEX_DIR, queries_dir=QUERIES_DIR,
    hybrid_in_path=OUT_PATH, out_path=RERANK_OUT_PATH,
)

In [ ]:
rerank_results = run_pageindex_hybrid_rerank_all(
    df=df, index_dir=INDEX_DIR, queries_dir=QUERIES_DIR,
    hybrid_in_path=OUT_PATH, out_path=RERANK_OUT_PATH,
)

### Did reranking win back the lost recall?

Compares, on the same questions: (a) narrowed hybrid (top 20, no rerank),
(b) narrowed hybrid + rerank (top 10), (c) vector RAG's own hybrid+rerank over
the whole document (`vector_rag_stage_rerank.jsonl`). (b) vs (c) is the
question this notebook exists to answer: does PageIndex narrowing help once
both sides get the same reranker?

In [34]:
PLAIN_RERANK_PATH = RESULTS_DIR / "vector_rag_stage_rerank.jsonl"

variants = {
    "narrowed hybrid (top20)":           _load_stage_records(OUT_PATH),
    "narrowed hybrid + rerank (top10)":  _load_stage_records(RERANK_OUT_PATH),
    "plain hybrid + rerank (top10)":     _load_stage_records(PLAIN_RERANK_PATH),
}
shared = set.intersection(*(set(v) for v in variants.values()))
print(f"Comparing on {len(shared)} questions present in all three\n")

rows = []
for name, recs in variants.items():
    row = {"variant": name}
    for k in K_VALUES:
        row[f"recall@{k}"] = sum(recs[i]["recall_at_k"][str(k)] for i in shared) / len(shared)
    for k in K_VALUES:
        row[f"mrr@{k}"] = sum(recs[i]["mrr_at_k"][str(k)] for i in shared) / len(shared)
    rows.append(row)
pd.DataFrame(rows).set_index("variant").round(3)

Comparing on 150 questions present in all three



,recall@5,recall@10,recall@15,mrr@5,mrr@10,mrr@15
variant,,,,,,
narrowed hybrid (top20),0.747,0.799,0.850,0.596,0.603,0.606
narrowed hybrid + rerank (top10),0.784,0.844,0.844,0.601,0.608,0.608
plain hybrid + rerank (top10),0.781,0.861,0.861,0.598,0.608,0.608


### Are the same questions failing in both approaches?

"Wrong" here = none of the gold pages made it into the reranked top 10
(`recall@10 == 0`). Labels every question (missed by both / only plain / only narrowed / right
in both) and shows all but "right in both", with gold pages, what each
approach returned and the reranker's scores. `nav_hit` says whether PageIndex navigation even
had a gold page in the pool for the narrowed approach.

In [35]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

narrowed = _load_stage_records(RERANK_OUT_PATH)          # narrowed hybrid + rerank
plain    = _load_stage_records(PLAIN_RERANK_PATH)        # vector RAG hybrid + rerank, whole doc
pool     = _load_stage_records(OUT_PATH)                 # for nav_hit (navigated_pages)
ids = sorted(set(narrowed) & set(plain))

MISS = lambda rec: rec["recall_at_k"]["10"] == 0         # change to "< 1" to also include partial misses

only_plain    = [i for i in ids if MISS(plain[i]) and not MISS(narrowed[i])]
only_narrowed = [i for i in ids if MISS(narrowed[i]) and not MISS(plain[i])]
both          = [i for i in ids if MISS(plain[i]) and MISS(narrowed[i])]
print(f"{len(ids)} questions compared")
print(f"  missed by both approaches : {len(both)}")
print(f"  missed only by plain      : {len(only_plain)}")
print(f"  missed only by narrowed   : {len(only_narrowed)}")
print(f"  right in both             : {len(ids) - len(both) - len(only_plain) - len(only_narrowed)}")

def top_pages_scores(rec, n=3):
    return [(pg, round(s, 3)) for pg, s in zip(rec["page_nums"][:n], rec["rerank_scores"][:n])]

def outcome(i):
    p_miss, n_miss = MISS(plain[i]), MISS(narrowed[i])
    if p_miss and n_miss: return "1. Missed by both"
    if p_miss:            return "2. Missed by plain only"
    if n_miss:            return "3. Missed by narrowed only"
    return "4. Right in both"

rows = []
for i in ids:
    q = df.loc[df.financebench_id == i].iloc[0]
    gold = sorted({e["evidence_page_num"] for e in q.evidence})
    rows.append({
        "outcome": outcome(i),
        "financebench_id": i, "doc_name": q.doc_name, "question_type": q.question_type,
        "question": q.question, "gold_pages": gold,
        "nav_hit": bool(set(gold) & set(pool[i]["navigated_pages"])) if i in pool else None,
        "plain_recall@10": plain[i]["recall_at_k"]["10"], "narrowed_recall@10": narrowed[i]["recall_at_k"]["10"],
        "plain_top3 (page, score)": top_pages_scores(plain[i]),
        "narrowed_top3 (page, score)": top_pages_scores(narrowed[i]),
        "plain_top10_pages": plain[i]["page_nums"],
        "narrowed_top10_pages": narrowed[i]["page_nums"],
    })
outcomes_df = pd.DataFrame(rows).sort_values(["outcome", "doc_name", "financebench_id"]).reset_index(drop=True)

# Everything except "right in both"
outcomes_df[outcomes_df["outcome"] != "4. Right in both"]

150 questions compared
  missed by both approaches : 14
  missed only by plain      : 4
  missed only by narrowed   : 7
  right in both             : 125


,outcome,financebench_id,doc_name,question_type,question,gold_pages,nav_hit,plain_recall@10,narrowed_recall@10,"plain_top3 (page, score)","narrowed_top3 (page, score)",plain_top10_pages,narrowed_top10_pages
0,1. Missed by both,financebench_id_01865,3M_2022_10K,novel-generated,"If we exclude the impact of M&A, which segment has dragged down 3M's overall growth in 2022?",[24],True,0.0,0.0,"[(3, 0.675), (31, 0.644), (27, 0.633)]","[(31, 0.644), (27, 0.633), (18, 0.626)]","[3, 31, 27, 18, 18, 32, 31, 30, 29, 60]","[31, 27, 18, 18, 32, 31, 30, 29, 60, 20]"
1,1. Missed by both,financebench_id_00540,AES_2022_10K,domain-relevant,Roughly how many times has AES Corporation sold its inventory in FY2022? Calculate inventory turnover ratio for the FY2022; if conventional inventory management is not meaningful for the company then state that and explain why.,"[129, 131]",True,0.0,0.0,"[(1, 0.626), (180, 0.544), (198, 0.521)]","[(180, 0.543), (214, 0.497), (213, 0.48)]","[1, 180, 198, 214, 213, 82, 208, 6, 126, 84]","[180, 214, 213, 82, 208, 6, 126, 179, 84, 210]"
2,1. Missed by both,financebench_id_01319,AES_2022_10K,domain-relevant,What is the quantity of restructuring costs directly outlined in AES Corporation's income statements for FY2022? If restructuring costs are not explicitly outlined then state 0.,[131],True,0.0,0.0,"[(1, 0.729), (180, 0.648), (216, 0.629)]","[(180, 0.647), (84, 0.61), (83, 0.528)]","[1, 180, 216, 210, 208, 84, 0, 82, 90, 210]","[180, 84, 83, 126, 82, 90, 106, 166, 181, 180]"
3,1. Missed by both,financebench_id_08135,AMAZON_2017_10K,metrics-generated,What is Amazon's year-over-year change in revenue from FY2016 to FY2017 (in units of percents and round to one decimal place)? Calculate what was asked by utilizing the line items clearly shown in the statement of income.,[37],True,0.0,0.0,"[(68, 0.649), (25, 0.609), (26, 0.596)]","[(68, 0.649), (25, 0.609), (26, 0.595)]","[68, 25, 26, 24, 17, 24, 71, 48, 31, 24]","[68, 25, 26, 24, 17, 24, 48, 71, 31, 24]"
4,1. Missed by both,financebench_id_00723,AMERICANEXPRESS_2022_10K,domain-relevant,"Does AMEX have an improving operating margin profile as of 2022? If operating margin is not a useful metric for a company like this, then state that and explain why.",[95],True,0.0,0.0,"[(62, 0.668), (44, 0.601), (152, 0.56)]","[(62, 0.668), (44, 0.601), (152, 0.56)]","[62, 44, 152, 45, 156, 42, 44, 91, 4, 52]","[62, 44, 152, 45, 42, 91, 44, 88, 52, 98]"
5,1. Missed by both,financebench_id_00685,BESTBUY_2023_10K,domain-relevant,"Are Best Buy's gross margins historically consistent (not fluctuating more than roughly 2% each year)? If gross margins are not a relevant metric for a company like this, then please state that and explain why.",[39],True,0.0,0.0,"[(3, 0.373), (25, 0.369), (43, 0.352)]","[(24, 0.583), (25, 0.37), (43, 0.35)]","[3, 25, 43, 22, 13, 68, 4, 25, 3, 7]","[24, 25, 43, 22, 26, 25, 23, 38, 49, 25]"
6,1. Missed by both,financebench_id_01091,BOEING_2022_10K,domain-relevant,Has Boeing reported any materially important ongoing legal battles from FY2022?,[112],True,0.0,0.0,"[(1, 0.601), (6, 0.595), (58, 0.55)]","[(55, 0.604), (54, 0.564), (58, 0.55)]","[1, 6, 58, 18, 15, 62, 131, 88, 118, 46]","[55, 54, 58, 56, 61, 18, 15, 62, 131, 88]"
7,1. Missed by both,financebench_id_01290,BOEING_2022_10K,domain-relevant,Who are the primary customers of Boeing as of FY2022?,"[7, 9, 13]",True,0.0,0.0,"[(113, 0.641), (114, 0.637), (115, 0.604)]","[(113, 0.641), (114, 0.637), (115, 0.604)]","[113, 114, 115, 114, 61, 23, 11, 82, 18, 2]","[113, 114, 115, 114, 61, 23, 11, 82, 18, 2]"
8,1. Missed by both,financebench_id_00790,CVSHEALTH_2022_10K,domain-relevant,Is CVS Health a capital-intensive business based on FY2022 data?,"[107, 109]",False,0.0,0.0,"[(110, 0.58), (88, 0.507), (113, 0.423)]","[(88, 0.507), (113, 0.422), (74, 0.37)]","[110, 88, 113, 74, 165, 118, 190, 13, 167, 114]","[88, 113, 74, 118, 127, 88, 117, 114, 75, 92]"
9,1. Missed by both,financebench_id_00299,JPMORGAN_2021Q1_10Q,

---
## Stage 7 — Generate answers from the reranked top-10 chunks

Same generation setup as the vectorless notebook, so the two are comparable:
same model (`gemini/gemini-3.1-flash-lite`), same `GENERATION_PROMPT`, same
JSON output (`reasoning` / `answer` / `cited_pages`), same 10 requests/minute
limit, and every call's tokens and latency logged to a cost file. The only
difference is *what* the generator is shown: the 10 reranked chunks from
Stage 6 (~512 tokens each) instead of whole PageIndex sections.

Each chunk is labeled `[Page N, Section: Chunk <id>]`, with `N` 1-indexed
(physical page) like the vectorless prompt. There is no selection-agent step
(vector RAG has one); the reranked top 10 goes straight to the generator.

**Costs API calls** (Gemini). Resumable.

In [36]:
import litellm, collections
from evaluation.cost_tracker import CostTracker

litellm.drop_params = True
MODEL = "gemini/gemini-3.1-flash-lite"     # same generation model as the vectorless notebook

GENERATION_OUT = RESULTS_DIR / "pageindex_hybrid_expanded_stage_generation.jsonl"
SCORING_OUT    = RESULTS_DIR / "pageindex_hybrid_expanded_stage_scoring.jsonl"
COSTS_OUT      = RESULTS_DIR / "pageindex_hybrid_expanded_costs.jsonl"
cost_tracker = CostTracker(COSTS_OUT)

_current = {"doc_name": None, "financebench_id": None, "stage": None}

REQUESTS_PER_MINUTE = 10                   # same cap as the vectorless notebook
_call_times = collections.deque()


def _wait_for_rate_limit():
    now = time.time()
    while _call_times and now - _call_times[0] > 60:
        _call_times.popleft()
    if len(_call_times) >= REQUESTS_PER_MINUTE:
        time.sleep(60 - (now - _call_times[0]))
    _call_times.append(time.time())


def llm_completion(model, prompt, max_retries=10):
    """litellm call + local token estimate + latency, logged per call (same as the vectorless wrapper)."""
    for attempt in range(max_retries):
        _wait_for_rate_limit()
        t0 = time.time()
        try:
            resp = litellm.completion(model=model, messages=[{"role": "user", "content": prompt}],
                                      drop_params=True, max_retries=0)
        except Exception as e:
            if attempt == max_retries - 1:
                raise
            print(f"    retrying after error ({e!s:.80s})")
            time.sleep(2 * (attempt + 1))
            continue
        content = resp.choices[0].message.content
        cost_tracker.log(
            pipeline="pageindex_hybrid", stage=_current["stage"] or "unknown", model=model.removeprefix("gemini/"),
            input_tokens=litellm.token_counter(model=model, text=prompt),
            output_tokens=litellm.token_counter(model=model, text=content or ""),
            doc_name=_current["doc_name"], financebench_id=_current["financebench_id"],
            latency_sec=time.time() - t0,
        )
        return content


# Identical to the vectorless notebook's prompt, so a difference in answers comes from the
# context handed to the generator, not from different instructions.
GENERATION_PROMPT = """You are a financial analyst answering a question using only the sections \
below from a company's SEC filing. Each section is labeled with its title and page number.

Preserve units and currency exactly as stated in the source (do not convert between millions and \
thousands, or between currencies). If the question requires a calculation across multiple figures, \
show the calculation briefly before giving the final answer. If the sections don't contain enough \
information to answer, say so explicitly rather than guessing.

Question: {question}

Sections:
{sections}

Respond in this exact JSON format:
{{
  "reasoning": "<brief reasoning, including any calculation and which section(s) you drew from>",
  "answer": "<the final answer, matching the format the question expects>",
  "cited_pages": [<page numbers you drew from>]
}}"""


def format_sections(sections: list[dict]) -> str:
    return "\n\n".join(f"[Page {s['page']}, Section: {s['title']}]\n{s['text']}" for s in sections)


def parse_json_answer(response):
    # Deviation from the vectorless notebook's bare json.loads: strip a ```json fence if the model added one.
    text = (response or "").strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    return json.loads(text)


def generate_answer(question, sections, model) -> dict:
    if not sections:
        return {"reasoning": "No sections retrieved.", "answer": "Insufficient information", "cited_pages": []}
    prompt = GENERATION_PROMPT.format(question=question, sections=format_sections(sections))
    return parse_json_answer(llm_completion(model, prompt))


def chunks_to_sections(rerank_rec, chunks_by_id):
    rows = chunks_by_id.loc[rerank_rec["chunk_ids"]].reset_index()   # keeps reranked order, best first
    return [{"title": f"Chunk {r.chunk_id}", "page": int(r.page_num) + 1, "text": r.text} for r in rows.itertuples()]


def run_generation_all(df, index_dir, rerank_path, out_path, model):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    rerank_records = _load_stage_records(rerank_path)
    completed = _load_completed_ids(out_path)
    doc_cache = {}

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            continue
        if fb_id not in rerank_records:
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED -- no rerank output yet")
            continue

        print(f"[{i}/{len(df)}] {fb_id}: generating ...")
        _current.update(doc_name=row.doc_name, financebench_id=fb_id, stage="generation")
        try:
            if row.doc_name not in doc_cache:
                chunks_df, _ = load_index(index_dir, row.doc_name)
                doc_cache[row.doc_name] = chunks_df.set_index("chunk_id")
            sections = chunks_to_sections(rerank_records[fb_id], doc_cache[row.doc_name])
            answer = generate_answer(row.question, sections, model)
            with out_path.open("a") as f:
                f.write(json.dumps({"financebench_id": fb_id, "doc_name": row.doc_name, "model_answer": answer, "timestamp": time.time()}) + "\n")
        except Exception as e:
            print(f"    FAILED: {e}")

### Try 5 questions first, then the full run

In [37]:
run_generation_all(df.head(5), INDEX_DIR, RERANK_OUT_PATH, GENERATION_OUT, MODEL)

for r in _load_jsonl(GENERATION_OUT):
    print(r["financebench_id"], "->", r["model_answer"])

[1/5] financebench_id_03029: generating ...
[2/5] financebench_id_04672: generating ...
[3/5] financebench_id_00499: generating ...
[4/5] financebench_id_01226: generating ...
[5/5] financebench_id_01865: generating ...
financebench_id_03029 -> {'reasoning': "According to the 'Cash Flows from Investing Activities' table and the 'Free Cash Flow (non-GAAP measure)' table, the purchase of property, plant and equipment (PP&E) for 2018 is explicitly stated as $1,577 million.", 'answer': '$1,577', 'cited_pages': [49, 60]}
financebench_id_04672 -> {'reasoning': "According to the Consolidated Balance Sheet on page 58, the line item 'Property, plant and equipment — net' for December 31, 2018, is reported as $8,738 million. Since the table states that all dollars are in millions, $8,738 million is equivalent to $8.738 billion.", 'answer': '8.738 USD billions', 'cited_pages': [58]}
financebench_id_00499 -> {'reasoning': "3M is a global manufacturer that maintains significant investments in long-t

In [38]:
run_generation_all(df, INDEX_DIR, RERANK_OUT_PATH, GENERATION_OUT, MODEL)

[6/150] financebench_id_00807: generating ...
[7/150] financebench_id_00941: generating ...
[8/150] financebench_id_01858: generating ...
[9/150] financebench_id_02987: generating ...
[10/150] financebench_id_07966: generating ...
[11/150] financebench_id_04735: generating ...
[12/150] financebench_id_07507: generating ...
[13/150] financebench_id_03856: generating ...
[14/150] financebench_id_00438: generating ...
[15/150] financebench_id_00591: generating ...
[16/150] financebench_id_01319: generating ...
[17/150] financebench_id_00540: generating ...
[18/150] financebench_id_10420: generating ...
[19/150] financebench_id_06655: generating ...
[20/150] financebench_id_08135: generating ...
[21/150] financebench_id_08286: generating ...
[22/150] financebench_id_03882: generating ...
[23/150] financebench_id_01935: generating ...
[24/150] financebench_id_00799: generating ...
[25/150] financebench_id_01079: generating ...
[26/150] financebench_id_01148: generating ...
[27/150] financeb

---
## Stage 8 — Score each answer

Same method as the vectorless notebook: `answer` is pulled out of the JSON;
`metrics-generated` questions go through the deterministic numeric match
first; anything not deterministically Correct (a deterministic miss, an
inconclusive match, or any other question type) goes to the LLM judge
(`openai/gpt-oss-120b` via Groq). Both verdicts are saved so they can be
compared. Uses the shared `evaluation/answer_scorer.py` (judge `max_tokens`
already raised to 1500 there).

Judge calls are free-tier (Groq) but are still token-logged. Resumable.

In [41]:
from groq import Groq
from evaluation.answer_scorer import score_deterministic, score_with_judge

judge_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None
JUDGE_MODEL = "openai/gpt-oss-120b"


def extract_answer_fields(model_answer):
    if isinstance(model_answer, dict):
        return str(model_answer.get("answer", "")), model_answer.get("reasoning"), model_answer.get("cited_pages")
    return str(model_answer), None, None


def run_scoring_all(df, generation_path, out_path, judge_client, judge_model):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    gen_records = {r["financebench_id"]: r for r in _load_jsonl(generation_path)}
    completed = _load_completed_ids(out_path)

    for i, row in enumerate(df.itertuples(), start=1):
        fb_id = row.financebench_id
        if fb_id in completed:
            continue
        if fb_id not in gen_records:
            print(f"[{i}/{len(df)}] {fb_id}: SKIPPED -- no generated answer yet")
            continue

        print(f"[{i}/{len(df)}] {fb_id}: scoring ...")
        _current.update(doc_name=row.doc_name, financebench_id=fb_id, stage="judge")
        try:
            answer_text, reasoning, cited_pages = extract_answer_fields(gen_records[fb_id]["model_answer"])
            is_metric = row.question_type == "metrics-generated"

            det_label, det_gold, det_matched = "NA", None, None
            if is_metric:
                det = score_deterministic(row.question, row.answer, answer_text)
                if det is None:
                    det_label = "Inconclusive"
                else:
                    det_label, det_gold, det_matched = det.label, det.gold_value, det.matched_value

            judge_label, judge_reasoning = None, None
            if det_label != "Correct":
                t0 = time.time()
                result, usage = score_with_judge(row.question, row.answer, answer_text, judge_client, judge_model)
                judge_label, judge_reasoning = result.label, result.reasoning
                cost_tracker.log(
                    pipeline="pageindex_hybrid", stage="judge", model=judge_model,
                    input_tokens=usage["input_tokens"], output_tokens=usage["output_tokens"],
                    doc_name=row.doc_name, financebench_id=fb_id, latency_sec=time.time() - t0,
                )

            record = {
                "financebench_id": fb_id, "doc_name": row.doc_name,
                "question_type": row.question_type, "question": row.question,
                "gold_answer": row.answer, "model_answer": answer_text,
                "model_reasoning": reasoning, "cited_pages": cited_pages,
                "deterministic_label": det_label,
                "deterministic_gold_value": det_gold, "deterministic_matched_value": det_matched,
                "judge_label": judge_label, "judge_reasoning": judge_reasoning,
                "label": "Correct" if det_label == "Correct" else judge_label,
                "method": "deterministic" if det_label == "Correct" else "llm_judge",
                "timestamp": time.time(),
            }
            with out_path.open("a") as f:
                f.write(json.dumps(record) + "\n")
        except Exception as e:
            print(f"    FAILED: {e}")


run_scoring_all(df, GENERATION_OUT, SCORING_OUT, judge_client, JUDGE_MODEL)

[1/150] financebench_id_03029: scoring ...
[2/150] financebench_id_04672: scoring ...
[3/150] financebench_id_00499: scoring ...
[4/150] financebench_id_01226: scoring ...
[5/150] financebench_id_01865: scoring ...
[6/150] financebench_id_00807: scoring ...
[7/150] financebench_id_00941: scoring ...
[8/150] financebench_id_01858: scoring ...
[9/150] financebench_id_02987: scoring ...
[10/150] financebench_id_07966: scoring ...
[11/150] financebench_id_04735: scoring ...
[12/150] financebench_id_07507: scoring ...
[13/150] financebench_id_03856: scoring ...
[14/150] financebench_id_00438: scoring ...
[15/150] financebench_id_00591: scoring ...
[16/150] financebench_id_01319: scoring ...
[17/150] financebench_id_00540: scoring ...
[18/150] financebench_id_10420: scoring ...
[19/150] financebench_id_06655: scoring ...
[20/150] financebench_id_08135: scoring ...
[21/150] financebench_id_08286: scoring ...
[22/150] financebench_id_03882: scoring ...
[23/150] financebench_id_01935: scoring .

---
## Stage 9 — Summarize and break down

Answer quality, retrieval hit rate (any gold page in the reranked top 10), and
tokens/cost by stage — then the per-question table (retrieval hit/miss vs.
answer correct/wrong, with question type and both scorer verdicts), then a
side-by-side with the vectorless pipeline.

In [42]:
from collections import Counter

scoring = _load_jsonl(SCORING_OUT)
n = len(scoring)
print(f"Answer quality ({n} scored questions):")
for label, count in Counter(r["label"] for r in scoring).items():
    print(f"  {label}: {count} ({100 * count / n:.1f}%)")

rerank_recs = _load_stage_records(RERANK_OUT_PATH)
hit = [rerank_recs[r["financebench_id"]]["recall_at_k"]["10"] > 0 for r in scoring if r["financebench_id"] in rerank_recs]
print(f"\nRetrieval hit rate (gold page in reranked top 10): {sum(hit) / len(hit):.3f}")

costs = _load_jsonl(COSTS_OUT)
tokens, dollars = Counter(), Counter()
for r in costs:
    tokens[r["stage"]] += r["input_tokens"] + r["output_tokens"]
    if r["cost_usd"] is not None:
        dollars[r["stage"]] += r["cost_usd"]
print(f"\nTokens/cost by stage ({len(costs)} logged calls):")
for stage, t in tokens.items():
    print(f"  {stage}: {t:,} tokens, ${dollars[stage]:.4f}")
print(f"  TOTAL: ${sum(dollars.values()):.4f}   (Jina rerank tokens are in the rerank file, unpriced)")

Answer quality (149 scored questions):
  Correct: 110 (73.8%)
  Incorrect: 25 (16.8%)
  Failure to Answer: 14 (9.4%)

Retrieval hit rate (gold page in reranked top 10): 0.859

Tokens/cost by stage (255 logged calls):
  generation: 671,265 tokens, $0.1992
  judge: 52,619 tokens, $0.0000
  TOTAL: $0.1992   (Jina rerank tokens are in the rerank file, unpriced)


In [ ]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

scores = pd.DataFrame(_load_jsonl(SCORING_OUT))
scores["retrieval_hit"] = scores["financebench_id"].map(lambda i: rerank_recs[i]["recall_at_k"]["10"] > 0)
scores["retrieval"] = scores["retrieval_hit"].map({True: "Retrieval hit", False: "Retrieval missed"})
scores["answer"] = scores["label"].map(lambda l: "Correct" if l == "Correct" else "Wrong")

def bucket(r):
    if r.retrieval == "Retrieval missed":
        return "1. Retrieval missed"
    return "2. Retrieved, answer wrong" if r.answer == "Wrong" else "3. Retrieved, answer correct"
scores["bucket"] = scores.apply(bucket, axis=1)
scores["judge_label"] = scores["judge_label"].fillna("not run")

print(pd.crosstab(scores["retrieval"], scores["answer"], margins=True))
print()
print(scores["bucket"].value_counts().sort_index().to_string())

cols = ["bucket", "financebench_id", "doc_name", "question_type", "question", "gold_answer", "model_answer",
        "deterministic_label", "deterministic_gold_value", "deterministic_matched_value",
        "judge_label", "judge_reasoning", "label"]
scores.sort_values(["bucket", "question_type", "doc_name"])[cols].reset_index(drop=True)

In [ ]:
# Side-by-side with the vectorless pipeline (whole PageIndex sections -> generator)
VECTORLESS_SCORING = RESULTS_DIR / "vectorless_rag_stage_scoring.jsonl"
VECTORLESS_COSTS   = RESULTS_DIR / "vectorless_rag_costs.jsonl"

vl = pd.DataFrame(_load_jsonl(VECTORLESS_SCORING)).set_index("financebench_id")
hy = scores.set_index("financebench_id")
shared = vl.index.intersection(hy.index)
print(f"{len(shared)} questions scored in both\n")

acc = lambda d: (d.loc[shared, "label"] == "Correct").mean()
print(f"Accuracy  vectorless (whole sections)  : {acc(vl):.3f}")
print(f"Accuracy  narrowed hybrid + rerank top10: {acc(hy):.3f}")

def gen_input_tokens_per_question(path):
    g = pd.DataFrame(_load_jsonl(path))
    g = g[g["stage"].isin(["generation"])]
    return g.groupby("financebench_id")["input_tokens"].sum()

vl_tok, hy_tok = gen_input_tokens_per_question(VECTORLESS_COSTS), gen_input_tokens_per_question(COSTS_OUT)
print(f"\nMedian generation input tokens / question: vectorless {vl_tok.median():,.0f}   hybrid {hy_tok.median():,.0f}")
print(f"Mean generation input tokens / question  : vectorless {vl_tok.mean():,.0f}   hybrid {hy_tok.mean():,.0f}")

flips = pd.DataFrame({
    "doc_name": hy.loc[shared, "doc_name"], "question_type": hy.loc[shared, "question_type"],
    "question": hy.loc[shared, "question"], "gold_answer": hy.loc[shared, "gold_answer"],
    "vectorless_answer": vl.loc[shared, "model_answer"], "vectorless_label": vl.loc[shared, "label"],
    "hybrid_answer": hy.loc[shared, "model_answer"], "hybrid_label": hy.loc[shared, "label"],
})
flips = flips[(flips.vectorless_label == "Correct") != (flips.hybrid_label == "Correct")]
print(f"\nQuestions where exactly one of the two is Correct: {len(flips)}")
flips.sort_values(["hybrid_label", "doc_name"])

---
## Stage 10 — Control: plain vector RAG's reranked top-10, straight to the generator

Vector RAG's own pipeline puts an LLM **selection agent** between rerank and
generation, which can drop a chunk the answer needed. This stage removes it:
vector RAG's reranked top 10 (`vector_rag_stage_rerank.jsonl`, whole-document
retrieval, no PageIndex narrowing) goes directly into the *same* generation
prompt and the *same* scoring method used for the narrowed run above.

Two comparisons come out of it:
- **Plain rerank-direct vs. narrowed rerank:** identical prompt, model and
  scoring, so any accuracy gap is down to retrieval (PageIndex narrowing).
- **Plain rerank-direct vs. vector RAG's original result:** the original used
  the selection agent *and* the older prompt and scoring rule, so a gap there
  mixes all three; it is not an isolated selection-agent effect.

Separate output and cost files, so nothing here mixes with the narrowed run's
files. Costs Gemini calls; resumable.

In [44]:
PLAIN_RERANK_PATH = RESULTS_DIR / "vector_rag_stage_rerank.jsonl"
PLAIN_GEN_OUT     = RESULTS_DIR / "plain_rerank_direct_stage_generation.jsonl"
PLAIN_SCORE_OUT   = RESULTS_DIR / "plain_rerank_direct_stage_scoring.jsonl"
PLAIN_COSTS_OUT   = RESULTS_DIR / "plain_rerank_direct_costs.jsonl"

# llm_completion / run_scoring_all log to the global `cost_tracker`. Point it at a separate file for this
# run so per-question token sums aren't doubled up with the narrowed run, then put it back.
cost_tracker = CostTracker(PLAIN_COSTS_OUT)
try:
    run_generation_all(df, INDEX_DIR, PLAIN_RERANK_PATH, PLAIN_GEN_OUT, MODEL)
    run_scoring_all(df, PLAIN_GEN_OUT, PLAIN_SCORE_OUT, judge_client, JUDGE_MODEL)
finally:
    cost_tracker = CostTracker(COSTS_OUT)

[28/150] financebench_id_01936: scoring ...
[53/150] financebench_id_01275: scoring ...
[106/150] financebench_id_01254: scoring ...


In [45]:
def load_labels(path, col="label"):
    return {r["financebench_id"]: r[col] for r in _load_jsonl(path)}

variants = {
    "vector RAG original (selection agent, old prompt/scoring)": load_labels(RESULTS_DIR / "vector_rag_stage_scoring.jsonl", "score_label"),
    "plain rerank top10 -> generator (new prompt/scoring)":       load_labels(PLAIN_SCORE_OUT),
    "narrowed hybrid + rerank top10 -> generator (same)":         load_labels(SCORING_OUT),
    "vectorless (whole PageIndex sections)":                      load_labels(RESULTS_DIR / "vectorless_rag_stage_scoring.jsonl"),
}
shared = set.intersection(*(set(v) for v in variants.values()))
print(f"{len(shared)} questions scored in all four\n")

rows = []
for name, labels in variants.items():
    c = Counter(labels[i] for i in shared)
    rows.append({"variant": name, "Correct": c["Correct"], "Incorrect": c["Incorrect"],
                 "Failure to Answer": c["Failure to Answer"], "accuracy": round(c["Correct"] / len(shared), 3)})
display(pd.DataFrame(rows).set_index("variant"))

def gen_input_tokens(path):
    g = pd.DataFrame(_load_jsonl(path))
    return g[g["stage"] == "generation"].groupby("financebench_id")["input_tokens"].sum()

print("Generation input tokens per question (median):")
for name, path in [("plain rerank direct", PLAIN_COSTS_OUT), ("narrowed rerank", COSTS_OUT),
                   ("vectorless", RESULTS_DIR / "vectorless_rag_costs.jsonl")]:
    print(f"  {name:<22} {gen_input_tokens(path).median():>10,.0f}")

# Same prompt/model/scoring, only retrieval differs: where do they disagree?
plain_l, narrow_l = variants["plain rerank top10 -> generator (new prompt/scoring)"], variants["narrowed hybrid + rerank top10 -> generator (same)"]
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)
plain_s = pd.DataFrame(_load_jsonl(PLAIN_SCORE_OUT)).set_index("financebench_id")
narrow_s = pd.DataFrame(_load_jsonl(SCORING_OUT)).set_index("financebench_id")
ids = sorted(i for i in shared if (plain_l[i] == "Correct") != (narrow_l[i] == "Correct"))
print(f"\nQuestions where exactly one of plain-rerank-direct / narrowed is Correct: {len(ids)}")
pd.DataFrame({
    "doc_name": [narrow_s.loc[i, "doc_name"] for i in ids], "question_type": [narrow_s.loc[i, "question_type"] for i in ids],
    "question": [narrow_s.loc[i, "question"] for i in ids], "gold_answer": [narrow_s.loc[i, "gold_answer"] for i in ids],
    "plain_answer": [plain_s.loc[i, "model_answer"] for i in ids], "plain_label": [plain_l[i] for i in ids],
    "narrowed_answer": [narrow_s.loc[i, "model_answer"] for i in ids], "narrowed_label": [narrow_l[i] for i in ids],
}, index=ids).sort_values(["narrowed_label", "doc_name"])

149 questions scored in all four



,Correct,Incorrect,Failure to Answer,accuracy
variant,,,,
"vector RAG original (selection agent, old prompt/scoring)",91,45,13,0.611
plain rerank top10 -> generator (new prompt/scoring),104,31,14,0.698
narrowed hybrid + rerank top10 -> generator (same),110,25,14,0.738
vectorless (whole PageIndex sections),118,28,3,0.792


Generation input tokens per question (median):
  plain rerank direct         4,520
  narrowed rerank             4,400
  vectorless                 55,540

Questions where exactly one of plain-rerank-direct / narrowed is Correct: 22


,doc_name,question_type,question,gold_answer,plain_answer,plain_label,narrowed_answer,narrowed_label
financebench_id_01226,3M_2022_10K,domain-relevant,"What drove operating margin change as of FY2022 for 3M? If operating margin is not a useful metric for a company like this, then please state that and explain why.","Operating Margin for 3M in FY2022 has decreased by 1.7% primarily due to: \n-Decrease in gross Margin\n-mostly one-off charges including Combat Arms Earplugs litigation, impairment related to exiting PFAS manufacturing, costs related to exiting Russia and divestiture-related restructuring\ncharges","The operating margin decreased from 20.8% in 2021 to 19.1% in 2022, a change of -1.7%. This decline was driven by several factors: increased costs for significant litigation (PFAS-related matters and Combat Arms Earplugs), higher raw material and logistics costs, manufacturing productivity headwinds (magnified by shutdowns in Belgium and restart activities), and reduced demand for disposable respirators (which negatively impacted margins by 0.3%). These negative pressures were partially offset by higher selling prices, ongoing spending discipline, and restructuring actions.",Incorrect,"As of FY2022, 3M's operating margin decreased by (1.7)%, moving from 20.8% in 2021 to 19.1% in 2022. This change was driven by several factors: negative impacts included increased special item costs for significant litigation (specifically PFAS-related matters and Combat Arms Earplugs litigation), higher raw material and logistics costs, manufacturing productivity headwinds, divestiture-related charges, and a 0.3% decline due to reduced disposable respirator demand. These negative pressures were partially offset by gains on business divestitures, selling price actions, spending discipline, restructuring benefits, and a 1.3% net benefit from organic growth/productivity and other impacts.",Correct
financebench_id_03856,ADOBE_2017_10K,metrics-generated,What is the FY2017 operating cash flow ratio for Adobe? Operating cash flow ratio is defined as: cash from operations / total current liabilities. Round your answer to two decimal places. Please utilize information provided primarily within the balance sheet and the cash flow statement.,0.83,0.82,Incorrect,0.83,Correct
financebench_id_01319,AES_2022_10K,domain-relevant,What is the quantity of restructuring costs directly outlined in AES Corporation's income statements for FY2022? If restructuring costs are not explicitly outlined then state 0.,0,"The provided sections do not contain enough information to determine the quantity of restructuring costs, as the Consolidated Statements of Operations are referenced in the Table of Contents but not provided in full detail within the excerpts.",Failure to Answer,0,Correct
financebench_id_01928,AMCOR_2023Q4_EARNINGS,novel-generated,What Was AMCOR's Adjusted Non GAAP EBITDA for FY 2023,"AMCOR's Adj. EBITDA was $2,018mn in FY 2023","$2,177 million",Incorrect,"2,018 million",Correct
financebench_id_00799,AMCOR_2023_10K,domain-relevant,"Has AMCOR's quick ratio improved or declined between FY2023 and FY2022? If the quick ratio is not something that a financial analyst would ask about a company like this, then state that and explain why.",The quick ratio has slightly improved from 0.67 times to 0.69 times between FY 2023 and FY 2022.(3.4% jump),AMCOR's quick ratio improved from 0.531 in FY2022 to 0.573 in FY2023.,Incorrect,AMCOR's quick ratio improved between FY2022 and FY2023.,Correct
financebench_id_03069,AMD_2015_10K,metrics-generated,"Answer the following question as if you are an equity research analyst and have lost internet connection so you do not have access to financial metric providers. According to the details clearly outlined within the P&L statement and the statement of cash flows, what is the FY2015 depreciation and amortization (D&A from cash flow statement) % margin for AMD?",4.2%,"The provided sections do not contain enough information to a

---
## Stage 11 — Clean cost breakdown (deduplicated)

Same issue as the other two notebooks: `COSTS_OUT` is append-only with no
"already logged this question" check, so any re-run of a generation/scoring
cell appends more records for the same question rather than replacing them.
Uses `evaluation/cost_tracker.py`'s `clean_stage_costs` to match each
question's cost record to the one that actually produced its saved answer
(nearest-timestamp join against `GENERATION_OUT` / `SCORING_OUT`), and
`recompute_cost_usd` to price everything fresh against the current pricing
table rather than a value frozen in at log time.

Jina rerank cost isn't in `COSTS_OUT` at all -- it was only ever saved as
`rerank_tokens` inside `RERANK_OUT_PATH`'s own (already one-row-per-question,
un-duplicated) records, so it's read from there directly and reported
unpriced (Jina has no confirmed rate).

Navigation and query expansion are **not** paid by this notebook -- it reuses
the vectorless notebook's expanded navigation (`NAVIGATION_PATH`, plus the
expansion call that fed it) and vector RAG's expanded queries (`QUERIES_DIR`).
They're still part of what it costs to answer a question this way, so they're
added as separate "borrowed" lines at the end and included in the full total --
kept on their own lines so a cross-pipeline sum doesn't count them twice.

In [ ]:
from evaluation.cost_tracker import clean_stage_costs, recompute_cost_usd

costs = _load_jsonl(COSTS_OUT)
gen_out = {r["financebench_id"]: r for r in _load_jsonl(GENERATION_OUT)}
scoring_all = _load_jsonl(SCORING_OUT)
judge_out = {r["financebench_id"]: r for r in scoring_all if r.get("judge_label") is not None}

n = len(gen_out)
print(f"Cleaning against {n} generated answers\n")

clean_records = []
for stage_name, out in [("generation", gen_out), ("judge", judge_out)]:
    kept, report = clean_stage_costs(costs, stage_name, out)
    print(f"{stage_name:<12} kept {report['n_kept']}/{report['n_input']} "
          f"(dropped {report['n_dropped_as_noise']} as noise)"
          + (f"  MISSING for: {report['missing_cost']}" if report["missing_cost"] else "")
          + (f"  late_only: {report['late_only']}" if report["late_only"] else ""))
    clean_records += kept
clean_records = recompute_cost_usd(clean_records)

by_stage = {}
for r in clean_records:
    d = by_stage.setdefault(r["stage"], {"tok": 0, "cost": 0.0, "unpriced": 0, "n": 0})
    d["tok"] += r["input_tokens"] + r["output_tokens"]; d["n"] += 1
    if r["cost_usd"] is not None:
        d["cost"] += r["cost_usd"]
    else:
        d["unpriced"] += 1

print("\nClean tokens/cost by stage:")
for stage, d in by_stage.items():
    unpriced_n = d["unpriced"]
    note = f"  ({unpriced_n} unpriced)" if unpriced_n else ""
    print(f"  {stage:<12} n={d['n']:<4} tokens={d['tok']:>9,}  ${d['cost']:.4f}{note}")

# Jina rerank: not in COSTS_OUT, read directly from its own (already clean) output file
rerank_recs_for_cost = _load_jsonl(RERANK_OUT_PATH)
rerank_tok = sum(r.get("rerank_tokens", 0) for r in rerank_recs_for_cost)
print(f"  {'rerank':<12} n={len(rerank_recs_for_cost):<4} tokens={rerank_tok:>9,}  $0.0000  (jina-reranker-v2-base-multilingual has no confirmed rate)")

llm_total = sum(d["cost"] for d in by_stage.values())
print(f"\nPer-query cost (generation + judge; rerank/navigation costed elsewhere or unpriced): "
      f"${llm_total:.4f} total  (${llm_total / n:.5f} / question, over {n} questions)")

# Borrowed stages -- work this pipeline reuses from the other two notebooks instead
# of paying for again, added so the total below is the hybrid's full
# question-to-answer cost, not just the stages this notebook ran itself:
#   - vectorless query_expansion + navigation_expanded: NAVIGATION_PATH's node picks,
#     which choose the pages hybrid retrieval is restricted to
#   - vector RAG query_expansion: the expanded query hybrid retrieval searches with
#     (QUERIES_DIR); its Stella embedding runs locally, so it's $0
# Reported on their own lines, not folded into the stages above, so summing costs
# across all three pipelines never counts the same API call twice.
from evaluation.cost_tracker import dedupe_by_last

vl_costs = _load_jsonl(RESULTS_DIR / "vectorless_rag_costs.jsonl")
vr_costs = _load_jsonl(RESULTS_DIR / "vector_rag_costs.jsonl")
nav_out = {fb_id: r for fb_id, r in _load_stage_records(NAVIGATION_PATH).items() if fb_id in gen_out}

borrowed = {}
for stage_name in ["query_expansion", "navigation_expanded"]:
    kept, report = clean_stage_costs(vl_costs, stage_name, nav_out)
    borrowed[f"vectorless {stage_name}"] = recompute_cost_usd(kept)
    if report["missing_cost"]:
        print(f"  vectorless {stage_name} MISSING for: {report['missing_cost']}")
borrowed["vector_rag query_expansion"] = recompute_cost_usd(
    [r for r in dedupe_by_last(vr_costs, "query_expansion") if r["financebench_id"] in gen_out])

print("\nBorrowed stages (paid by the other notebooks, reused here):")
borrowed_total = 0.0
for label, recs in borrowed.items():
    tok = sum(r["input_tokens"] + r["output_tokens"] for r in recs)
    cost = sum(r["cost_usd"] or 0.0 for r in recs)
    unpriced_n = sum(r["cost_usd"] is None for r in recs)
    note = f"  ({unpriced_n} unpriced)" if unpriced_n else ""
    print(f"  {label:<31} n={len(recs):<4} tokens={tok:>9,}  ${cost:.4f}{note}")
    borrowed_total += cost

full_total = llm_total + borrowed_total
print(f"\nFull per-query cost (own + borrowed): ${full_total:.4f} total  "
      f"(${full_total / n:.5f} / question, over {n} questions)")

---
## Stage 12 — Dedicated latency pass

The generation/rerank cells above are independently resumable (each stage
runs to completion for all 150 questions before the next starts), so
per-question end-to-end latency can't be read off them -- two stages for the
same question can run hours apart. Same fix as vector RAG's Stage 8 and
vectorless's equivalent pass: a separate, sequential, uncached timing pass
over a fixed sample, each question timed `LATENCY_REPEATS` times back to
back. N=10, repeats=3, matching both other notebooks so all three are
directly comparable.

Times: restrict-to-navigated-pages + hybrid retrieval, rerank, generation.
Navigation itself isn't timed here (it's vectorless's cost/latency to
report, reused here for free) -- if you want *this pipeline's* full
question-to-answer latency including navigation, add vectorless's per-call
navigation latency to this stage's total, don't re-time it.

In [ ]:
import statistics

LATENCY_SAMPLE_SIZE = 10   # matches vector RAG's Stage 8 and vectorless's latency pass
LATENCY_REPEATS = 3
HYBRID_LATENCY_PATH = RESULTS_DIR / "pageindex_hybrid_expanded_latency.jsonl"

nav_records_all = _load_stage_records(NAVIGATION_PATH)


def time_single_hybrid_pass(row, tree, node_map, chunks_df, dense_embeddings, model):
    timings = {}
    t_total0 = time.time()

    t0 = time.time()
    node_ids = nav_records_all[row.financebench_id]["node_ids"]
    page_nums = navigated_pages(node_ids, node_map)
    restricted_df, restricted_embeddings = restrict_to_pages(chunks_df, dense_embeddings, page_nums)
    if len(restricted_df) == 0:
        restricted_df, restricted_embeddings = chunks_df, dense_embeddings
    bm25_index = build_bm25(restricted_df)
    _, meta_path = query_paths(QUERIES_DIR, row.financebench_id)
    expanded_query = json.loads(meta_path.read_text())["query_text"]
    query_vec = np.load(query_paths(QUERIES_DIR, row.financebench_id)[0])
    top_hybrid = hybrid_retrieve(restricted_df, restricted_embeddings, bm25_index, query_vec, expanded_query, top_k=20)
    timings["hybrid_retrieval"] = time.time() - t0

    t0 = time.time()
    top_rerank, _ = rerank(JINA_API_KEY, expanded_query, top_hybrid, top_k=10)
    timings["rerank"] = time.time() - t0

    t0 = time.time()
    sections = chunks_to_sections({"chunk_ids": top_rerank["chunk_id"].tolist()}, chunks_df.set_index("chunk_id"))
    answer = generate_answer(row.question, sections, model)
    timings["generation"] = time.time() - t0

    timings["total"] = time.time() - t_total0
    return timings, answer


def run_hybrid_latency_pass(df, out_path, sample_size=LATENCY_SAMPLE_SIZE, repeats=LATENCY_REPEATS, seed=42):
    out_path.parent.mkdir(parents=True, exist_ok=True)
    navigated = df[df.financebench_id.isin(nav_records_all)]
    sample = navigated.sample(n=min(sample_size, len(navigated)), random_state=seed)

    completed_pairs = {(r["financebench_id"], r["repeat"]) for r in _load_jsonl(out_path)}
    doc_cache = {}
    for row in sample.itertuples():
        if row.doc_name not in doc_cache:
            tree = load_tree(TREE_DIR_FLASH, row.doc_name)
            node_map = create_node_mapping(tree["structure"])
            chunks_df, dense_embeddings = load_index(INDEX_DIR, row.doc_name)
            doc_cache[row.doc_name] = (tree, node_map, chunks_df, dense_embeddings)
        tree, node_map, chunks_df, dense_embeddings = doc_cache[row.doc_name]

        for repeat in range(1, repeats + 1):
            if (row.financebench_id, repeat) in completed_pairs:
                continue
            print(f"  {row.financebench_id} repeat {repeat}/{repeats} ...")
            try:
                timings, answer = time_single_hybrid_pass(row, tree, node_map, chunks_df, dense_embeddings, MODEL)
                with out_path.open("a") as f:
                    f.write(json.dumps({"financebench_id": row.financebench_id, "doc_name": row.doc_name,
                                        "repeat": repeat, "timings_sec": timings, "model_answer": answer}) + "\n")
            except Exception as e:
                print(f"    FAILED: {e}")


def summarize_hybrid_latency(out_path):
    records = _load_jsonl(out_path)
    if not records:
        print(f"No latency data yet in {out_path}")
        return {}
    by_question = {}
    for r in records:
        by_question.setdefault(r["financebench_id"], []).append(r["timings_sec"]["total"])
    per_question_median = {fb_id: statistics.median(vals) for fb_id, vals in by_question.items()}
    overall_median = statistics.median(per_question_median.values())
    stage_names = [k for k in records[0]["timings_sec"] if k != "total"]
    print(f"Median end-to-end latency ({len(per_question_median)} questions, navigation excluded -- see note above): {overall_median:.2f}s")
    for stage in stage_names:
        med = statistics.median(r["timings_sec"][stage] for r in records)
        print(f"  median {stage}: {med:.2f}s")
    return {"overall_median_sec": overall_median}

In [ ]:
# Runs the timing pass -- real API calls; skips (question, repeat) pairs already timed
run_hybrid_latency_pass(df, HYBRID_LATENCY_PATH)

In [ ]:
# Summary only -- reads the saved latency file, makes no API calls
summarize_hybrid_latency(HYBRID_LATENCY_PATH)

---
## Stage 13 — Variant: rerank the whole document first, then keep only chunks on navigated pages

The narrowed hybrid above applies PageIndex **before** retrieval: it restricts the chunk pool to the
navigated pages, then runs hybrid retrieval + rerank inside it. This variant flips the order:

1. Take vector RAG's own whole-document hybrid retrieval + Jina rerank top 10 (`vector_rag_stage_rerank.jsonl`),
   whose recall is already high (gold page in the top 10 for 132/150 questions).
2. Keep only the top-10 chunks whose page is among the pages PageIndex navigation picked
   (`NAVIGATION_PATH`, same expanded navigation as everything else here), in reranked order.
3. Generate and score exactly like Stage 10's control: same prompt, model, JSON format and scoring rule.

**Question:** PageIndex acting as a *filter* on the reranked chunks cuts how many chunks (and tokens)
reach the generator. Does that hurt accuracy, or does dropping off-topic chunks help?

Compared against **Stage 10's control** (the same top 10, unfiltered), which makes this a paired
comparison where the only difference is the filter. If no chunk survives the filter, the question falls back to
the full top 10 so it's never answered from nothing. On this data that never happens.

Separate output and cost files (`rerank_then_filter_*`), so nothing mixes with the other runs.
Costs Gemini calls (~150, small prompts); resumable.

In [ ]:
PLAIN_RERANK_PATH  = RESULTS_DIR / "vector_rag_stage_rerank.jsonl"         # Stage 10's inputs/outputs, redefined
PLAIN_SCORE_OUT    = RESULTS_DIR / "plain_rerank_direct_stage_scoring.jsonl"  # so this stage runs on its own
PLAIN_COSTS_OUT    = RESULTS_DIR / "plain_rerank_direct_costs.jsonl"
FILTER_RERANK_PATH = RESULTS_DIR / "rerank_then_filter_stage_rerank.jsonl"
FILTER_GEN_OUT     = RESULTS_DIR / "rerank_then_filter_stage_generation.jsonl"
FILTER_SCORE_OUT   = RESULTS_DIR / "rerank_then_filter_stage_scoring.jsonl"
FILTER_COSTS_OUT   = RESULTS_DIR / "rerank_then_filter_costs.jsonl"


def build_filtered_rerank(plain_rerank_path, navigation_path, tree_dir, out_path):
    """Keep only the reranked chunks whose page PageIndex navigation selected, in reranked order.
    If none survive, fall back to the full top 10 (flagged). Deterministic, so the file is
    simply rewritten on every run."""
    plain, nav = _load_stage_records(plain_rerank_path), _load_stage_records(navigation_path)
    node_maps, rows = {}, []
    for fb_id, r in plain.items():
        if r["doc_name"] not in node_maps:
            node_maps[r["doc_name"]] = create_node_mapping(load_tree(tree_dir, r["doc_name"])["structure"])
        pages = navigated_pages(nav[fb_id]["node_ids"], node_maps[r["doc_name"]])
        keep = [(c, p) for c, p in zip(r["chunk_ids"], r["page_nums"]) if p in pages]
        rows.append({"financebench_id": fb_id, "doc_name": r["doc_name"],
                     "chunk_ids": [c for c, _ in keep] or r["chunk_ids"],
                     "page_nums": [p for _, p in keep] or r["page_nums"],
                     "n_before_filter": len(r["chunk_ids"]), "fell_back_to_top10": not keep})
    out_path.write_text("".join(json.dumps(r) + "\n" for r in rows))
    return pd.DataFrame(rows).set_index("financebench_id")


filtered = build_filtered_rerank(PLAIN_RERANK_PATH, NAVIGATION_PATH, TREE_DIR_FLASH, FILTER_RERANK_PATH)
gold = {r.financebench_id: {e["evidence_page_num"] for e in r.evidence} for r in df.itertuples()}
plain_rerank = _load_stage_records(PLAIN_RERANK_PATH)
gold_before = sum(bool(set(plain_rerank[i]["page_nums"]) & gold[i]) for i in filtered.index)
gold_after = sum(bool(set(filtered.loc[i, "page_nums"]) & gold[i]) for i in filtered.index)

print(f"Chunks passed on: mean {filtered.chunk_ids.str.len().mean():.1f} of 10 "
      f"(fell back to full top 10 for {filtered.fell_back_to_top10.sum()} questions)")
print(f"Gold page among the chunks: {gold_before} questions before the filter, {gold_after} after")
filtered.chunk_ids.str.len().value_counts().sort_index().rename("questions").rename_axis("chunks kept").to_frame().T

In [ ]:
# Same pattern as Stage 10: point the global cost_tracker at this run's own file, then put it back
cost_tracker = CostTracker(FILTER_COSTS_OUT)
try:
    run_generation_all(df, INDEX_DIR, FILTER_RERANK_PATH, FILTER_GEN_OUT, MODEL)
    run_scoring_all(df, FILTER_GEN_OUT, FILTER_SCORE_OUT, judge_client, JUDGE_MODEL)
finally:
    cost_tracker = CostTracker(COSTS_OUT)

In [ ]:
from math import comb


def mcnemar_exact(a: pd.Series, b: pd.Series):
    """Two-sided exact McNemar test on paired correct/not-correct outcomes."""
    only_a, only_b = int((a & ~b).sum()), int((~a & b).sum())
    n, k = only_a + only_b, min(only_a, only_b)
    return only_a, only_b, (min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n) if n else 1.0)


def labels_of(path):
    return pd.Series({i: r["label"] for i, r in _load_stage_records(path).items()}).reindex(df.financebench_id)


def median_gen_tokens(cost_path):
    g = pd.DataFrame(_load_jsonl(cost_path))
    g = g[g.stage == "generation"].sort_values("timestamp").groupby("financebench_id").last()   # latest call per question
    return g.input_tokens.median()


runs = {
    "Control: whole-doc rerank top 10 (Stage 10)":  (PLAIN_SCORE_OUT, PLAIN_COSTS_OUT),
    "Rerank top 10, filtered to navigated pages":    (FILTER_SCORE_OUT, FILTER_COSTS_OUT),
    "Narrowed first, then retrieve + rerank":        (SCORING_OUT, COSTS_OUT),
}
labels = {name: labels_of(score) for name, (score, _) in runs.items()}
control = labels["Control: whole-doc rerank top 10 (Stage 10)"].eq("Correct")

rows = []
for name, (score_path, cost_path) in runs.items():
    l = labels[name]
    only_this, only_control, p = mcnemar_exact(l.eq("Correct"), control)
    rows.append({"run": name, "Correct": int(l.eq("Correct").sum()), "Failure to Answer": int(l.eq("Failure to Answer").sum()),
                 "Accuracy": f"{l.eq('Correct').mean():.1%}", "Median generation input tokens": int(median_gen_tokens(cost_path)),
                 "Only this correct": only_this, "Only control correct": only_control,
                 "p vs control (McNemar)": round(p, 3) if name != next(iter(runs)) else None})
pd.DataFrame(rows).set_index("run")

### Where do the filtered run and the control disagree?

Questions answered correctly by exactly one of the two. Worth reading: whether the filter's losses are the few questions whose gold page it removed.

In [ ]:
pd.set_option("display.max_colwidth", None)
f_lab, c_lab = labels["Rerank top 10, filtered to navigated pages"], labels["Control: whole-doc rerank top 10 (Stage 10)"]
f_s, c_s = _load_stage_records(FILTER_SCORE_OUT), _load_stage_records(PLAIN_SCORE_OUT)
diff_ids = [i for i in df.financebench_id if (f_lab[i] == "Correct") != (c_lab[i] == "Correct")]
pd.DataFrame([{
    "financebench_id": i, "question": f_s[i]["question"], "gold_answer": f_s[i]["gold_answer"],
    "control_answer": c_s[i]["model_answer"], "control_label": c_lab[i],
    "filtered_answer": f_s[i]["model_answer"], "filtered_label": f_lab[i],
    "chunks kept": len(filtered.loc[i, "chunk_ids"]),
    "gold page kept by filter": bool(set(filtered.loc[i, "page_nums"]) & gold[i]),
} for i in diff_ids]).set_index("financebench_id").sort_values("filtered_label")

---
## Sync results back to GitHub

Same pattern as the other three pipelines. `.gitignore` has a
`experiments/results/*` blanket ignore with explicit `!` exceptions per file
— `pageindex_hybrid_stage_retrieval.jsonl` and `pageindex_hybrid_errors.log`
were added there alongside this notebook so this sync doesn't silently no-op.

In [ ]:
!git -C "{REPO_ROOT}" add experiments/results/pageindex_hybrid_expanded_stage_retrieval.jsonl experiments/results/pageindex_hybrid_expanded_stage_rerank.jsonl experiments/results/pageindex_hybrid_expanded_stage_generation.jsonl experiments/results/pageindex_hybrid_expanded_stage_scoring.jsonl experiments/results/pageindex_hybrid_expanded_costs.jsonl experiments/results/plain_rerank_direct_stage_generation.jsonl experiments/results/plain_rerank_direct_stage_scoring.jsonl experiments/results/plain_rerank_direct_costs.jsonl experiments/results/pageindex_hybrid_expanded_latency.jsonl experiments/results/pageindex_hybrid_errors.log experiments/results/rerank_then_filter_stage_rerank.jsonl experiments/results/rerank_then_filter_stage_generation.jsonl experiments/results/rerank_then_filter_stage_scoring.jsonl experiments/results/rerank_then_filter_costs.jsonl
!git -C "{REPO_ROOT}" commit -m "Sync PageIndex-guided hybrid retrieval results" || echo "(nothing new to commit)"
!git -C "{REPO_ROOT}" push origin main